# Engage Corpus Evaluation - v3.5 Data

Scores data using the single-label format as is used in the original paper

**A key problem solved in this script**:
- Predictions contain ALL users (train + test): 33,727 users
- Ground truth only contains TEST users: 27,896 users

**The Solution**:
- Extract test user indices from test.tsv
- Filter predictions to only include test users
- Evaluate using filtered predictions

## 1. Setup and Imports

In [ ]:
import numpy as np
import pickle
from pathlib import Path
from typing import Dict, Tuple
import pandas as pd

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

## 3. Configuration

In [ ]:
# ============================================================
# PATHS - Update BASE_DIR to match your Google Drive structure
# ============================================================

BASE_DIR = Path('/content/drive/MyDrive/CIS 5300/predictions')
DATA_DIR = Path('/content/drive/MyDrive/CIS 5300/engage_corpus_processed_v3_5tfidf')

# Ground truth file
GROUND_TRUTH_FILE = BASE_DIR / 'ground_truth_v35_test.npy'

# Test data file (to extract test user indices)
TEST_FILE = DATA_DIR / 'ncf_data' / 'test.tsv'

# Prediction files
PREDICTION_FILES = {
    'Cosine Similarity (v3.5, 128-dim)': BASE_DIR / 'cosine_v35_128.npy',
    'Cosine Similarity (v3.5, 4096-dim)': BASE_DIR / 'cosine_v35_4096.npy',
    'NCF Model A (v3.5)': BASE_DIR / 'predictions_model_A.npy',
    'NCF Model D (v3.5 Gated)': BASE_DIR / 'predictions_model_D.npy', # New
}

K = 10

## 4. Extract Test User Indices (KEY FIX)

In [ ]:
def get_test_user_indices(test_file: Path) -> np.ndarray:
    """
    Extract unique user indices from test.tsv file.

    This function is key to fixing the dimension mismatch mentioned at the beginning:
    - Predictions are generated for ALL users (train + test)
    - Ground truth only has TEST users
    - We need to know which indices in predictions correspond to test users

    Returns:
        Array of test user indices (sorted)
    """
    test_users = []
    with open(test_file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                test_users.append(int(parts[0]))

    return np.array(sorted(set(test_users)), dtype=np.int32)


print("Extracting test user indices...")
test_user_indices = get_test_user_indices(TEST_FILE)

print(f"Number of unique test users: {len(test_user_indices)}")
print(f"Min user index: {test_user_indices.min()}")
print(f"Max user index: {test_user_indices.max()}")
print(f"Sample indices: {test_user_indices[:10]}")

## 5. Evaluation Functions

In [ ]:
def hit_rate_at_k(predictions: np.ndarray, ground_truth: np.ndarray, k: int = 10) -> float:
    n_users = predictions.shape[0]
    hits = 0

    for i in range(n_users):
        top_k_items = np.argsort(predictions[i])[::-1][:k]
        if ground_truth[i] in top_k_items:
            hits += 1

    return hits / n_users


def ndcg_at_k(predictions: np.ndarray, ground_truth: np.ndarray, k: int = 10) -> float:
    n_users = predictions.shape[0]
    ndcg_sum = 0.0

    for i in range(n_users):
        top_k_items = np.argsort(predictions[i])[::-1][:k]

        try:
            rank = np.where(top_k_items == ground_truth[i])[0][0] + 1
            dcg = 1.0 / np.log2(rank + 1)
            ndcg_sum += dcg
        except IndexError:
            pass

    return ndcg_sum / n_users


def evaluate_predictions(predictions: np.ndarray, ground_truth: np.ndarray, k: int = 10) -> Dict[str, float]:
    hr = hit_rate_at_k(predictions, ground_truth, k)
    ndcg = ndcg_at_k(predictions, ground_truth, k)

    return {
        f'hr@{k}': hr,
        f'ndcg@{k}': ndcg
    }

## 6. Load Ground Truth

In [ ]:
print(f"Loading ground truth from: {GROUND_TRUTH_FILE}")
ground_truth = np.load(GROUND_TRUTH_FILE)

print(f"Ground truth shape: {ground_truth.shape}")
print(f"Ground truth dtype: {ground_truth.dtype}")
print(f"Number of test users: {len(ground_truth)}")

# Validate against extracted test user indices
if len(test_user_indices) != len(ground_truth):
    print(f"\n  WARNING: Mismatch!")
    print(f"   Test users from file: {len(test_user_indices)}")
    print(f"   Ground truth entries: {len(ground_truth)}")
else:
    print(f"\n✓ Validation passed: {len(ground_truth)} test users match")

## 7. Evaluate All Models (WITH FIX)

In [ ]:
results = {}

print("=" * 80)
print("ENGAGE CORPUS EVALUATION - v3.5 DATA")
print("=" * 80)
print(f"\nEvaluating metrics: HR@{K} and NDCG@{K}")
print(f"Number of test users: {len(ground_truth)}\n")

for model_name, pred_file in PREDICTION_FILES.items():
    print(f"\n{"="*80}")
    print(f"Model: {model_name}")
    print(f"File: {pred_file.name}")
    print("-" * 80)

    try:
        # Load ALL predictions (including train users)
        predictions_all = np.load(pred_file)
        print(f"Full predictions shape: {predictions_all.shape}")

        # KEY FIX: Filter to only test users
        predictions_test = predictions_all[test_user_indices]
        print(f"Test predictions shape: {predictions_test.shape}")

        # Validate shapes
        if predictions_test.shape[0] != len(ground_truth):
            print(f"  ERROR: Shape mismatch after filtering!")
            print(f"   Predictions: {predictions_test.shape[0]} users")
            print(f"   Ground truth: {len(ground_truth)} users")
            continue

        # Evaluate
        print(f"\nEvaluating...")
        model_results = evaluate_predictions(predictions_test, ground_truth, k=K)
        results[model_name] = model_results

        # Print results
        print(f"\n✓ Results:")
        print(f"  Hit Rate @ {K}:  {model_results[f'hr@{K}']:.4f}  ({model_results[f'hr@{K}']*100:.2f}%)")
        print(f"  NDCG @ {K}:      {model_results[f'ndcg@{K}']:.4f}")

    except FileNotFoundError:
        print(f" ERROR: File not found: {pred_file}")
    except Exception as e:
        print(f" ERROR: {str(e)}")

print(f"\n{"="*80}")

## 8. Summary Table

In [ ]:
if results:
    summary_data = []
    for model_name, metrics in results.items():
        summary_data.append({
            'Model': model_name,
            f'HR@{K}': f"{metrics[f'hr@{K}']:.4f} ({metrics[f'hr@{K}']*100:.2f}%)",
            f'NDCG@{K}': f"{metrics[f'ndcg@{K}']:.4f}"
        })

    summary_df = pd.DataFrame(summary_data)

    print("\n" + "="*80)
    print("SUMMARY - v3.5 DATA")
    print("="*80 + "\n")
    print(summary_df.to_string(index=False))
    print("\n" + "="*80)

    display(summary_df)

    # Best models
    best_hr = max(results.items(), key=lambda x: x[1][f'hr@{K}'])
    best_ndcg = max(results.items(), key=lambda x: x[1][f'ndcg@{K}'])

    print("\n BEST MODELS:\n")
    print(f"Best HR@{K}: {best_hr[0]}")
    print(f"  Score: {best_hr[1][f'hr@{K}']:.4f} ({best_hr[1][f'hr@{K}']*100:.2f}%)\n")
    print(f"Best NDCG@{K}: {best_ndcg[0]}")
    print(f"  Score: {best_ndcg[1][f'ndcg@{K}']:.4f}")
else:
    print("\n  No results to display")